# UAH-DriveSet: Model Training and Evaluation
Run AFTER uah_driveset_pipeline.py (needs uah_features.csv in the same folder).
pip install scikit-learn imbalanced-learn joblib

In [ ]:
# %% Cell 1: Imports, configuration and load features

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    LeaveOneGroupOut
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.neural_network import MLPClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


# =========================================================
# REPRODUCIBILITY
# =========================================================

SEED = 42

np.random.seed(SEED)


# =========================================================
# GENERAL SETTINGS
# =========================================================

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

os.makedirs("plots", exist_ok=True)
os.makedirs("models", exist_ok=True)


# =========================================================
# LOAD FEATURE TABLE
# =========================================================

df = pd.read_csv("uah_features.csv")


META = [
    "trip",
    "driver",
    "behavior",
    "road",
    "label",
    "label_name"
]


FEATURE_COLS = [
    c for c in df.columns
    if c not in META
]


X = df[FEATURE_COLS].values
y = df["label"].values


groups_trip = df["trip"].values
groups_driver = df["driver"].values


NAMES = [
    "Normal",
    "Braking",
    "Turning",
    "Acceleration"
]


# =========================================================
# DISPLAY INFORMATION
# =========================================================

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Feature table:", df.shape)

print("Number of features:", len(FEATURE_COLS))

print("\nLabel distribution:")
print(df["label_name"].value_counts())

print("\nRandom seed:", SEED)

In [ ]:
# %% Cell 2: Pipeline and model definitions


def make_pipe(clf, smote=True):
    """
    Create a machine-learning pipeline.

    StandardScaler:
        Standardizes the input features.

    SMOTE:
        Handles class imbalance using synthetic oversampling.
        It is applied only to the training data when the
        pipeline is fitted.
    """

    steps = [
        ("scale", StandardScaler())
    ]

    if smote:

        steps.append(
            (
                "smote",
                SMOTE(
                    k_neighbors=3,
                    random_state=SEED
                )
            )
        )

    steps.append(
        ("clf", clf)
    )

    return Pipeline(steps)


# =========================================================
# MODELS
# =========================================================

MODELS = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000
        ),

    "KNN":
        KNeighborsClassifier(
            n_neighbors=7
        ),

    "SVM (RBF)":
        SVC(
            kernel="rbf"
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            n_jobs=-1,
            random_state=SEED
        ),

    "Gradient Boosting":
        HistGradientBoostingClassifier(
            random_state=SEED
        ),

    "MLP Neural Net":
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=200,
            random_state=SEED
        )
}


print("Models:")

for name in MODELS:
    print("-", name)


# =========================================================
# RISKY EVENT RECALL
# =========================================================

def risky_recall(y_true, y_pred):

    """
    Calculates recall for detecting any risky event.

    Normal      = 0
    Braking     = 1
    Turning     = 2
    Acceleration= 3

    Risky event = anything other than Normal.
    """

    true_risky = y_true != 0
    pred_risky = y_pred != 0

    return recall_score(
        true_risky,
        pred_risky,
        zero_division=0
    )


In [ ]:
# %% Cell 3: Model comparison
#
# 3-fold GroupKFold by trip.
#
# Windows belonging to the same trip are kept together.
#
# SMOTE is applied only to the training portion of each fold.


print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)


gkf = GroupKFold(
    n_splits=3
)


results = []


for name, clf in MODELS.items():

    accs = []
    f1s = []
    risky_recalls = []

    print("\nTraining:", name)


    for fold, (tr, te) in enumerate(
        gkf.split(
            X,
            y,
            groups_trip
        ),
        start=1
    ):

        pipe = make_pipe(
            clf,
            smote=True
        )


        pipe.fit(
            X[tr],
            y[tr]
        )


        pred = pipe.predict(
            X[te]
        )


        acc = accuracy_score(
            y[te],
            pred
        )


        macro_f1 = f1_score(
            y[te],
            pred,
            average="macro",
            zero_division=0
        )


        risky = risky_recall(
            y[te],
            pred
        )


        accs.append(acc)
        f1s.append(macro_f1)
        risky_recalls.append(risky)


        print(
            f"  Fold {fold}: "
            f"Accuracy={acc:.3f}, "
            f"Macro-F1={macro_f1:.3f}, "
            f"Risky Recall={risky:.3f}"
        )


    results.append({

        "Model": name,

        "Accuracy":
            np.mean(accs),

        "Macro-F1":
            np.mean(f1s),

        "Risky-event recall":
            np.mean(risky_recalls)

    })


    print("  done:", name)


# =========================================================
# RESULTS TABLE
# =========================================================

res = pd.DataFrame(
    results
).set_index(
    "Model"
).round(3)


print("\n" + "=" * 60)
print("MODEL COMPARISON RESULTS")
print("=" * 60)

print(res)


# =========================================================
# COMPARISON GRAPH
# =========================================================

res.plot(
    kind="bar",
    figsize=(10, 5)
)


plt.ylim(0, 1)

plt.title(
    "Model comparison (3-fold GroupKFold by trip)"
)

plt.ylabel("Score")

plt.xticks(
    rotation=25,
    ha="right"
)

plt.tight_layout()


plt.savefig(
    "plots/08_model_comparison.png",
    dpi=150
)


plt.show()

In [ ]:
# %% Cell 4: Class imbalance experiment
#
# Compare:
# 1. No handling
# 2. Class weights
# 3. SMOTE


print("\n" + "=" * 60)
print("CLASS IMBALANCE EXPERIMENT")
print("=" * 60)


rf_kw = {
    "n_estimators": 200,
    "n_jobs": -1,
    "random_state": SEED
}


setups = {

    "No handling":
        make_pipe(
            RandomForestClassifier(
                **rf_kw
            ),
            smote=False
        ),


    "Class weights":
        make_pipe(
            RandomForestClassifier(
                class_weight="balanced",
                **rf_kw
            ),
            smote=False
        ),


    "SMOTE":
        make_pipe(
            RandomForestClassifier(
                **rf_kw
            ),
            smote=True
        )
}


rows = []


for name, pipe in setups.items():

    accs = []
    f1s = []
    risky_recalls = []

    print("\nStrategy:", name)


    for fold, (tr, te) in enumerate(
        gkf.split(
            X,
            y,
            groups_trip
        ),
        start=1
    ):

        pipe.fit(
            X[tr],
            y[tr]
        )


        pred = pipe.predict(
            X[te]
        )


        accs.append(
            accuracy_score(
                y[te],
                pred
            )
        )


        f1s.append(
            f1_score(
                y[te],
                pred,
                average="macro",
                zero_division=0
            )
        )


        risky_recalls.append(
            risky_recall(
                y[te],
                pred
            )
        )


    rows.append({

        "Strategy": name,

        "Accuracy":
            np.mean(accs),

        "Macro-F1":
            np.mean(f1s),

        "Risky-event recall":
            np.mean(risky_recalls)

    })


# =========================================================
# RESULTS
# =========================================================

imb = pd.DataFrame(
    rows
).set_index(
    "Strategy"
).round(3)


print("\nImbalance experiment results:")

print(imb)


# =========================================================
# GRAPH
# =========================================================

imb.plot(
    kind="bar",
    figsize=(8, 4)
)


plt.ylim(0, 1)

plt.title(
    "Effect of imbalance handling"
)

plt.ylabel("Score")

plt.xticks(
    rotation=0
)

plt.tight_layout()


plt.savefig(
    "plots/09_imbalance_experiment.png",
    dpi=150
)


plt.show()

In [ ]:
# %% Cell 5: Final model
#
# 75% of trips = training
# 25% of trips = completely unseen test trips
#
# Final model:
# Random Forest + SMOTE


print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)


# =========================================================
# TRAIN / TEST SPLIT
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)


tr, te = next(
    gss.split(
        X,
        y,
        groups_trip
    )
)


print(
    "Training windows:",
    len(tr)
)

print(
    "Testing windows:",
    len(te)
)

print(
    "Training trips:",
    len(np.unique(groups_trip[tr]))
)

print(
    "Testing trips:",
    len(np.unique(groups_trip[te]))
)


# =========================================================
# FINAL RANDOM FOREST
# =========================================================

final_rf = RandomForestClassifier(
    **rf_kw
)


final = make_pipe(
    final_rf,
    smote=True
)


final.fit(
    X[tr],
    y[tr]
)


pred = final.predict(
    X[te]
)


# =========================================================
# CLASSIFICATION REPORT
# =========================================================

labels_present = sorted(
    np.unique(
        np.concatenate(
            [y[te], pred]
        )
    )
)


print("\nClassification Report:\n")


print(
    classification_report(
        y[te],
        pred,
        labels=labels_present,
        target_names=[
            NAMES[i]
            for i in labels_present
        ],
        zero_division=0
    )
)


# =========================================================
# OVERALL METRICS
# =========================================================

final_accuracy = accuracy_score(
    y[te],
    pred
)


final_macro_f1 = f1_score(
    y[te],
    pred,
    average="macro",
    zero_division=0
)


final_risky_recall = risky_recall(
    y[te],
    pred
)


print(
    "Overall test accuracy:",
    round(final_accuracy, 3)
)

print(
    "Test macro-F1:",
    round(final_macro_f1, 3)
)

print(
    "Risky-event recall:",
    round(final_risky_recall, 3)
)


# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y[te],
    pred,
    labels=labels_present,
    normalize="true"
)


fig, ax = plt.subplots(
    figsize=(6, 5)
)


sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    xticklabels=[
        NAMES[i]
        for i in labels_present
    ],
    yticklabels=[
        NAMES[i]
        for i in labels_present
    ]
)


ax.set_xlabel("Predicted")

ax.set_ylabel("True")

ax.set_title(
    "Normalized confusion matrix"
)


plt.tight_layout()


plt.savefig(
    "plots/10_confusion_matrix.png",
    dpi=150
)


plt.show()


# =========================================================
# FEATURE IMPORTANCE
# =========================================================

rf_model = final.named_steps["clf"]


importance = pd.Series(
    rf_model.feature_importances_,
    index=FEATURE_COLS
)


top_features = (
    importance
    .sort_values()
    .tail(15)
)


print("\nTop 15 features:")

print(
    top_features.sort_values(
        ascending=False
    )
)


top_features.plot(
    kind="barh",
    figsize=(7, 6)
)


plt.title(
    "Top 15 feature importances"
)

plt.xlabel(
    "Importance"
)


plt.tight_layout()


plt.savefig(
    "plots/11_feature_importance.png",
    dpi=150
)


plt.show()


# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(
    {
        "model": final,
        "features": FEATURE_COLS,
        "seed": SEED
    },
    "models/final_model.joblib"
)


print(
    "\nSaved: models/final_model.joblib"
)

In [ ]:
# %% Cell 6: Driving-risk score


print("\n" + "=" * 60)
print("DRIVING RISK SCORE")
print("=" * 60)


# =========================================================
# PREDICT PROBABILITIES
# =========================================================

proba = final.predict_proba(
    X[te]
)


classes = list(
    final.named_steps["clf"].classes_
)


# Probability of Normal class

if 0 in classes:

    p_normal = proba[
        :,
        classes.index(0)
    ]

else:

    p_normal = np.zeros(
        len(te)
    )


# =========================================================
# RISK SCORE
# =========================================================

score = 100 * (
    1 - p_normal
)


level = pd.cut(
    score,

    bins=[
        -1,
        33,
        66,
        101
    ],

    labels=[
        "Low",
        "Medium",
        "High"
    ]
)


risk = pd.DataFrame({

    "score": score,

    "level": level,

    "behavior":
        df["behavior"].values[te]

})


# =========================================================
# DISPLAY RESULTS
# =========================================================

print(
    "\nRisk score statistics by behavior:"
)


print(
    risk
    .groupby(
        "behavior",
        observed=True
    )["score"]
    .describe()
    .round(1)
)


print(
    "\nRisk level counts by behavior:"
)


print(
    pd.crosstab(
        risk["behavior"],
        risk["level"]
    )
)


# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(6, 4)
)


sns.boxplot(
    data=risk,
    x="behavior",
    y="score",
    ax=ax,
    showfliers=False
)


ax.set_title(
    "Driving-risk score by driving behavior "
    "(test trips)"
)


ax.set_ylabel(
    "Risk score (0-100)"
)


ax.set_xlabel(
    "Driving behavior"
)


plt.tight_layout()


plt.savefig(
    "plots/12_risk_score.png",
    dpi=150
)


plt.show()

In [ ]:
# %% Cell 7: Leave-one-driver-out evaluation
#
# Each driver is held out once.
#
# The model is trained using the other drivers and tested
# entirely on the unseen driver.


print("\n" + "=" * 60)
print("LEAVE-ONE-DRIVER-OUT EVALUATION")
print("=" * 60)


logo = LeaveOneGroupOut()


rows = []


for tr, te in logo.split(
    X,
    y,
    groups_driver
):

    test_driver = groups_driver[te][0]


    print(
        f"\nTest driver: {test_driver}"
    )


    # -----------------------------------------------------
    # Train model
    # -----------------------------------------------------

    driver_model = make_pipe(
        RandomForestClassifier(
            **rf_kw
        ),
        smote=True
    )


    driver_model.fit(
        X[tr],
        y[tr]
    )


    # -----------------------------------------------------
    # Predict
    # -----------------------------------------------------

    p = driver_model.predict(
        X[te]
    )


    # -----------------------------------------------------
    # Metrics
    # -----------------------------------------------------

    acc = accuracy_score(
        y[te],
        p
    )


    macro_f1 = f1_score(
        y[te],
        p,
        average="macro",
        zero_division=0
    )


    risky = risky_recall(
        y[te],
        p
    )


    rows.append({

        "Test driver":
            test_driver,

        "Accuracy":
            acc,

        "Macro-F1":
            macro_f1,

        "Risky-event recall":
            risky
    })


    print(
        f"Accuracy: {acc:.3f}"
    )

    print(
        f"Macro-F1: {macro_f1:.3f}"
    )

    print(
        f"Risky-event recall: {risky:.3f}"
    )


# =========================================================
# RESULTS TABLE
# =========================================================

lodo = pd.DataFrame(
    rows
).set_index(
    "Test driver"
).round(3)


print(
    "\n" + "=" * 60
)

print(
    "LEAVE-ONE-DRIVER-OUT RESULTS"
)

print(
    "=" * 60
)


print(lodo)


# =========================================================
# MEAN RESULTS
# =========================================================

print("\nMean:")

print(
    lodo.mean().round(3)
)


# =========================================================
# GRAPH
# =========================================================

lodo.plot(
    kind="bar",
    figsize=(8, 4)
)


plt.ylim(
    0,
    1
)


plt.ylabel(
    "Score"
)


plt.title(
    "Leave-one-driver-out results"
)


plt.xticks(
    rotation=0
)


plt.tight_layout()


plt.savefig(
    "plots/13_leave_one_driver_out.png",
    dpi=150
)


plt.show()